# 01 – Exploratory Data Analysis (EDA)

This notebook downloads NASDAQ-100 data, inspects its structure, and creates
exploratory visualisations to understand the data before modelling.

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from src.config import TICKER, START_DATE, END_DATE
from src.data.collector import DataCollector
from src.visualization.plotter import Plotter

print(f'Ticker : {TICKER}')
print(f'Period : {START_DATE} → {END_DATE}')

Ticker : ^NDX
Period : 2020-01-01 → 2025-12-31


In [2]:
# ── 1. Download stock data ──────────────────────────────────────────────────
collector = DataCollector()
df = collector.download_stock_data(save=True)
print(f'Shape : {df.shape}')
df.head()

Shape : (1507, 6)


Price,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2020-01-02,8872.219727,8872.219727,8873.629883,8786.900391,8802.219727,2862700000
2020-01-03,8793.900391,8793.900391,8843.650391,8755.169922,8755.169922,2586520000
2020-01-06,8848.519531,8848.519531,8849.980469,8713.889648,8713.889648,2810450000
2020-01-07,8846.450195,8846.450195,8872.469727,8821.679688,8857.139648,2381740000
2020-01-08,8912.370117,8912.370117,8953.549805,8834.940430,8845.450195,2472620000


In [3]:
# ── 2. Basic statistics ─────────────────────────────────────────────────────
df.describe()

Price,Adj Close,Close,High,Low,Open,Volume
count,1507.000000,1507.000000,1507.000000,1507.000000,1507.000000,1.507000e+03
mean,15545.591601,15545.591601,15658.134324,15416.787902,15541.958865,5.622449e+09
std,4363.279450,4363.279450,4371.038544,4352.044606,4366.900406,1.943482e+09
min,6994.290039,6994.290039,7145.290039,6771.910156,6952.709961,2.169020e+09
25%,12250.370117,12250.370117,12339.090332,12063.680176,12216.120117,4.392050e+09
50%,14786.360352,14786.360352,14907.509766,14698.290039,14772.290039,5.031850e+09
75%,18725.275391,18725.275391,18888.634766,18580.290039,18729.189453,6.300085e+09
max,26119.849609,26119.849609,26182.099609,25907.449219,26147.720703,1.630873e+10


In [4]:
# ── 3. Missing values ───────────────────────────────────────────────────────
print('Missing values:')
print(df.isnull().sum())

Missing values:
Price
Adj Close    0
Close        0
High         0
Low          0
Open         0
Volume       0
dtype: int64


In [15]:
plotter = Plotter()
plotter.plot_price_history(
    df_plot,
    title=f"{TICKER} Closing Price",
    filename="price_history.png",
    xlim=(pd.Timestamp(START_DATE), pd.Timestamp(END_DATE)),
)

In [6]:
# ── 5. Returns distribution ─────────────────────────────────────────────────
returns = df['Close'].pct_change().dropna()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(returns.index, returns, linewidth=0.5, color='steelblue')
axes[0].set_title('Daily Returns')
axes[0].set_ylabel('Return')
axes[0].grid(alpha=0.3)

axes[1].hist(returns, bins=80, color='steelblue', edgecolor='white')
axes[1].set_title('Return Distribution')
axes[1].set_xlabel('Daily Return')
axes[1].grid(alpha=0.3)

plt.tight_layout()
fig.savefig('../reports/figures/returns_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Skewness : {returns.skew():.4f}')
print(f'Kurtosis : {returns.kurtosis():.4f}')

Skewness : -0.1176
Kurtosis : 7.1699


In [7]:
# ── 6. Rolling volatility (30-day) ──────────────────────────────────────────
vol = returns.rolling(30).std() * (252 ** 0.5)  # annualised

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(vol.index, vol, color='tomato', linewidth=0.8)
ax.set_title('30-Day Rolling Annualised Volatility')
ax.set_ylabel('Volatility')
ax.grid(alpha=0.3)
fig.autofmt_xdate()
fig.savefig('../reports/figures/rolling_volatility.png', dpi=150, bbox_inches='tight')
plt.show()

In [8]:
# ── 7. Volume analysis ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(df.index, df['Volume'], width=1, color='grey', alpha=0.6)
ax.set_title('Daily Trading Volume')
ax.set_ylabel('Volume')
ax.grid(alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
fig.savefig('../reports/figures/trading_volume.png', dpi=150, bbox_inches='tight')
plt.show()